In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from statsmodels.stats.diagnostic import compare_encompassing
from scipy import stats

In [2]:
# Step 1: Import and reshape raw WDI panel data

# Read the raw file
wdi_raw = pd.read_csv("../data/panel_raw_wdi_2000_2022.csv")

# Replace placeholder missing values ".." with NaN
wdi_raw = wdi_raw.replace("..", np.nan)

# Melt the year columns (2000–2022) into a single Year column
wdi_long = wdi_raw.melt(
    id_vars=["Country Name", "Country Code", "Series Name", "Series Code"],
    var_name="Year",
    value_name="Value"
)

# Clean up the Year column to retain only the numeric year (e.g., "2000" from "2000 [YR2000]")
wdi_long["Year"] = wdi_long["Year"].str.extract(r"(\d{4})").astype(int)

# Convert Value to numeric
wdi_long["Value"] = pd.to_numeric(wdi_long["Value"], errors="coerce")

# Quick check of the structure
print("Shape after melting:", wdi_long.shape)
print(wdi_long.head(10))

Shape after melting: (55062, 6)
  Country Name Country Code  \
0  Afghanistan          AFG   
1  Afghanistan          AFG   
2  Afghanistan          AFG   
3  Afghanistan          AFG   
4  Afghanistan          AFG   
5  Afghanistan          AFG   
6  Afghanistan          AFG   
7  Afghanistan          AFG   
8  Afghanistan          AFG   
9      Albania          ALB   

                                         Series Name           Series Code  \
0  Carbon dioxide (CO2) emissions excluding LULUC...  EN.GHG.CO2.PC.CE.AR5   
1                 GDP per capita (constant 2015 US$)        NY.GDP.PCAP.KD   
2       Energy use (kg of oil equivalent per capita)     EG.USE.PCAP.KG.OE   
3           Urban population (% of total population)     SP.URB.TOTL.IN.ZS   
4                                   Trade (% of GDP)        NE.TRD.GNFS.ZS   
5  Renewable energy consumption (% of total final...        EG.FEC.RNEW.ZS   
6  Industry (including construction), value added...        NV.IND.TOTL.ZS   
7 

In [3]:
# Step 2A: Pivot indicators so each variable becomes its own column

# Pivot the data: each Series Code becomes a column
wdi = wdi_long.pivot_table(
    index=["Country Name", "Country Code", "Year"],
    columns="Series Code",
    values="Value"
).reset_index()

# Flatten the multi-level column index created by pivot_table
wdi.columns.name = None

# Optional: Rename columns for clarity using a dictionary
rename_map = {
    "EN.GHG.CO2.PC.CE.AR5": "co2_per_capita",
    "NY.GDP.PCAP.KD": "gdp_per_capita",
    "EG.USE.PCAP.KG.OE": "energy_use_per_capita",
    "SP.URB.TOTL.IN.ZS": "urban_pct",
    "NE.TRD.GNFS.ZS": "trade_pct_gdp",
    "EG.FEC.RNEW.ZS": "renewable_pct",
    "NV.IND.TOTL.ZS": "industry_value_added_pct_gdp",
    "EG.USE.COMM.FO.ZS": "fossil_fuel_pct",
    "EN.POP.DNST": "population_density"
}
wdi.rename(columns=rename_map, inplace=True)

# Sort the dataset for neatness
wdi = wdi.sort_values(["Country Name", "Year"]).reset_index(drop=True)

# Check results
print("Shape after pivot:", wdi.shape)
print(wdi.head(10))

Shape after pivot: (6084, 12)
  Country Name Country Code  Year  renewable_pct  fossil_fuel_pct  \
0  Afghanistan          AFG  2000           45.0              NaN   
1  Afghanistan          AFG  2001           45.6              NaN   
2  Afghanistan          AFG  2002           37.8              NaN   
3  Afghanistan          AFG  2003           36.7              NaN   
4  Afghanistan          AFG  2004           44.2              NaN   
5  Afghanistan          AFG  2005           33.9              NaN   
6  Afghanistan          AFG  2006           31.9              NaN   
7  Afghanistan          AFG  2007           28.8              NaN   
8  Afghanistan          AFG  2008           21.2              NaN   
9  Afghanistan          AFG  2009           16.5              NaN   

   energy_use_per_capita  co2_per_capita  population_density  trade_pct_gdp  \
0                    NaN        0.050476           30.863847            NaN   
1                    NaN        0.046573           3

In [4]:
# Step 2B: Keep only codes that are true ISO-3 country codes recognized by WDI
print("Shape before filtering countries:", wdi.shape)
print(f"Unique countries in the WDI dataset before filtering to valid ISO-3 codes: {wdi['Country Name'].nunique()}")
wdi_countries = wdi[wdi["Country Code"].isin(
    pd.read_csv("https://raw.githubusercontent.com/datasets/country-codes/master/data/country-codes.csv")["ISO3166-1-Alpha-3"]
)]
print("Shape after filtering countries:", wdi_countries.shape)
print(f"Unique countries in the WDI dataset after filtering to valid ISO-3 codes: {wdi_countries['Country Name'].nunique()}")

# Countries dropped
print("Countries dropped during filtering:", set(wdi["Country Name"]) - set(wdi_countries["Country Name"]))

Shape before filtering countries: (6084, 12)
Unique countries in the WDI dataset before filtering to valid ISO-3 codes: 265
Shape after filtering countries: (4934, 12)
Unique countries in the WDI dataset after filtering to valid ISO-3 codes: 215
Countries dropped during filtering: {'OECD members', 'Kosovo', 'Middle East, North Africa, Afghanistan & Pakistan (excluding high income)', 'Pre-demographic dividend', 'Heavily indebted poor countries (HIPC)', 'Middle East, North Africa, Afghanistan & Pakistan (IDA & IBRD)', 'Latin America & the Caribbean (IDA & IBRD countries)', 'Latin America & Caribbean', 'Arab World', 'North America', 'Europe & Central Asia', 'High income', 'South Asia', 'Upper middle income', 'Other small states', 'Europe & Central Asia (excluding high income)', 'Sub-Saharan Africa', 'Sub-Saharan Africa (IDA & IBRD countries)', 'Late-demographic dividend', 'Low income', 'Fragile and conflict affected situations', 'Central Europe and the Baltics', 'Africa Western and Centra

In [5]:
# Step 3: Create log-transformed and derived variables for panel regression

panel_df = wdi_countries.copy()

log_vars = {
    "co2_per_capita": "lnCO2",
    "gdp_per_capita": "lnGDP",
    "energy_use_per_capita": "lnEnergy",
    "urban_pct": "lnUrban",
    "trade_pct_gdp": "lnTrade",
    "population_density": "lnPopDensity"
}

# Replace zero or negative values with NaN before taking logs
for src, tgt in log_vars.items():
    panel_df[src] = panel_df[src].apply(lambda x: np.nan if (pd.isna(x) or x <= 0) else x)
    panel_df[tgt] = np.log(panel_df[src])

# Add squared GDP term
panel_df["lnGDP_sq"] = panel_df["lnGDP"] ** 2

# Keep Renewable, FossilFuel, Industry as levels
level_vars = ["renewable_pct", "fossil_fuel_pct", "industry_value_added_pct_gdp"]

# Drop rows missing any core variables
core_vars = list(log_vars.values()) + ["lnGDP_sq"] + level_vars
panel_df = panel_df.dropna(subset=core_vars)

print("✅ Transformed dataset ready for regression (zeros safely handled).")
print("Shape after dropping missing rows:", panel_df.shape)
display(panel_df[["Country Name", "Year"] + core_vars].head(5))

✅ Transformed dataset ready for regression (zeros safely handled).
Shape after dropping missing rows: (2906, 19)


,Country Name,Year,lnCO2,lnGDP,lnEnergy,lnUrban,lnTrade,lnPopDensity,lnGDP_sq,renewable_pct,fossil_fuel_pct,industry_value_added_pct_gdp
69,Albania,2000,0.045400,7.596851,6.365396,3.731484,4.120812,4.725068,57.712152,41.4,58.626225,21.712185
70,Albania,2001,0.116797,7.691163,6.393972,3.747973,4.162742,4.715684,59.153987,39.0,60.857697,24.386650
71,Albania,2002,0.272420,7.739406,6.493932,3.772784,4.189525,4.712685,59.898412,35.8,63.130793,26.238327
72,Albania,2003,0.326016,7.795107,6.475984,3.797128,4.171664,4.708943,60.763694,33.7,64.410168,27.084688
73,Albania,2004,0.377229,7.850609,6.574550,3.821026,4.174971,4.704764,61.632064,35.8,65.467668,27.729975


In [6]:
# Step 4: Baseline pooled OLS regression

# Define dependent and independent variables
y = panel_df["lnCO2"]
X = panel_df[[
    "lnGDP", "lnGDP_sq", "lnEnergy", "lnUrban", "lnTrade",
    "renewable_pct", "fossil_fuel_pct", "lnPopDensity"
]]

# Add constant
X = sm.add_constant(X)

# Fit model
pooled_ols = sm.OLS(y, X).fit()

print("✅ Baseline pooled OLS completed.")
print(pooled_ols.summary())


✅ Baseline pooled OLS completed.
                            OLS Regression Results                            
Dep. Variable:                  lnCO2   R-squared:                       0.946
Model:                            OLS   Adj. R-squared:                  0.946
Method:                 Least Squares   F-statistic:                     6371.
Date:                Fri, 31 Oct 2025   Prob (F-statistic):               0.00
Time:                        23:39:29   Log-Likelihood:                -991.96
No. Observations:                2906   AIC:                             2002.
Df Residuals:                    2897   BIC:                             2056.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const    

In [7]:
# Step 5A: Hausman Test - Fixed vs Random Effects

# 1. Ensure panel structure exists
panel_df = panel_df.set_index(["Country Name", "Year"])

# 2. Define dependent and independent variables
y = panel_df["lnCO2"]
X = panel_df[[
    "lnGDP", "lnGDP_sq", "lnEnergy", "lnUrban", "lnTrade",
    "renewable_pct", "fossil_fuel_pct", "lnPopDensity"
]]

# 3. Estimate Random Effects model (no explicit constant)
re_model = RandomEffects(y, X).fit(cov_type="clustered", cluster_entity=True)
print("✅ Random Effects model estimated.")
print(re_model.summary)

# 4. Estimate Fixed Effects model (needed for comparison)
fe_model_temp = PanelOLS(y, X, entity_effects=True).fit(cov_type="clustered", cluster_entity=True)

# 5. Manual Hausman Test
b_fe = fe_model_temp.params
b_re = re_model.params
v_fe = fe_model_temp.cov
v_re = re_model.cov

common_coef = list(set(b_fe.index) & set(b_re.index))
b_diff = b_fe[common_coef] - b_re[common_coef]
v_diff = v_fe.loc[common_coef, common_coef] - v_re.loc[common_coef, common_coef]

hausman_stat = float(b_diff.T @ np.linalg.inv(v_diff) @ b_diff)
df = len(common_coef)
p_value = 1 - stats.chi2.cdf(hausman_stat, df)

print("\nHausman test statistic:", round(hausman_stat, 3))
print("Degrees of freedom:", df)
print("p-value:", round(p_value, 5))

if p_value < 0.05:
    print("❌ Reject H0: Random Effects inconsistent → use Fixed Effects.")
else:
    print("✅ Fail to reject H0: Random Effects acceptable.")


✅ Random Effects model estimated.
                        RandomEffects Estimation Summary                        
Dep. Variable:                  lnCO2   R-squared:                        0.7076
Estimator:              RandomEffects   R-squared (Between):              0.8054
No. Observations:                2906   R-squared (Within):               0.6989
Date:                Fri, Oct 31 2025   R-squared (Overall):              0.9119
Time:                        23:39:29   Log-likelihood                    2053.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      876.48
Entities:                         157   P-value                           0.0000
Avg Obs:                       18.510   Distribution:                  F(8,2898)
Min Obs:                       1.0000                                           
Max Obs:                       23.000   F-statistic (robust):             8

In [8]:
# Step 5B: Country fixed-effects panel regression

# Define dependent and independent variables
y = panel_df["lnCO2"]
X = panel_df[[
    "lnGDP", "lnGDP_sq", "lnEnergy", "lnUrban", "lnTrade",
    "renewable_pct", "fossil_fuel_pct", "lnPopDensity"
]]

# Add constant (intercept handled separately by entity_effects=True)
X = sm.add_constant(X)

# Fit the fixed-effects (within) model with country effects
fe_model = PanelOLS(y, X, entity_effects=True).fit(cov_type="clustered", cluster_entity=True)

print("✅ Country Fixed-Effects model estimated.")
print(fe_model.summary)

✅ Country Fixed-Effects model estimated.
                          PanelOLS Estimation Summary                           
Dep. Variable:                  lnCO2   R-squared:                        0.7659
Estimator:                   PanelOLS   R-squared (Between):              0.8083
No. Observations:                2906   R-squared (Within):               0.7659
Date:                Fri, Oct 31 2025   R-squared (Overall):              0.9435
Time:                        23:39:29   Log-likelihood                    2498.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1121.1
Entities:                         157   P-value                           0.0000
Avg Obs:                       18.510   Distribution:                  F(8,2741)
Min Obs:                       1.0000                                           
Max Obs:                       23.000   F-statistic (robust):       

In [9]:
# Step 6: Country and Time Fixed Effects regression

# Time dummies
time_dummies = pd.get_dummies(panel_df.index.get_level_values("Year"), drop_first=True)
X_timefe = pd.concat([X, time_dummies.set_index(X.index)], axis=1)

# Fit FE model with both entity and time effects
fe_time_model = PanelOLS(y, X_timefe, entity_effects=True).fit(cov_type="clustered", cluster_entity=True)

print("✅ Country + Time Fixed Effects model estimated.")
print(fe_time_model.summary)


✅ Country + Time Fixed Effects model estimated.
                          PanelOLS Estimation Summary                           
Dep. Variable:                  lnCO2   R-squared:                        0.7750
Estimator:                   PanelOLS   R-squared (Between):              0.7984
No. Observations:                2906   R-squared (Within):               0.7750
Date:                Fri, Oct 31 2025   R-squared (Overall):              0.9359
Time:                        23:39:29   Log-likelihood                    2555.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      312.16
Entities:                         157   P-value                           0.0000
Avg Obs:                       18.510   Distribution:                 F(30,2719)
Min Obs:                       1.0000                                           
Max Obs:                       23.000   F-statistic (robust):

In [10]:
panel_df.index.get_level_values("Country Name").nunique()

157